# 01 - Ingest TMDB titles
POC 31: CineScope. Pulls in-scope movies and TV series from the TMDB
`/discover` endpoints, year by year, and writes the `raw_titles` Delta table
to the lakehouse.

Scope cut (agreed): `vote_count >= 500`, first release 1950 onwards,
movies and TV.

This product uses the TMDB API but is not endorsed or certified by TMDB.
TMDB caching terms: refresh or delete stored data within 6 months.

**Prerequisites**
- A free TMDB API key (https://www.themoviedb.org/settings/api)
- Key stored in a Fabric workspace setting or pasted below for the POC run
- A lakehouse attached as the default lakehouse of this notebook


In [ ]:
# Configuration
# For the POC the key is read from an environment-style variable cell.
# Do NOT commit a real key to GitHub. Production option: Azure Key Vault
# via notebookutils.credentials.getSecret().

TMDB_API_KEY = ""  # paste key here for the POC run, clear before commit

VOTE_COUNT_MIN = 500
YEAR_FROM = 1950
YEAR_TO = 2026

assert TMDB_API_KEY, "Set TMDB_API_KEY before running"


In [ ]:
import requests, time

BASE = "https://api.themoviedb.org/3"
session = requests.Session()
session.params = {"api_key": TMDB_API_KEY}

def tmdb_get(path, **params):
    # Polite client: honour 429 Retry-After, retry transient errors.
    for attempt in range(5):
        r = session.get(f"{BASE}{path}", params=params, timeout=30)
        if r.status_code == 429:
            time.sleep(float(r.headers.get("Retry-After", 1)) + 0.1)
            continue
        if r.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"TMDB request failed after retries: {path}")


In [ ]:
# Genre id -> name maps (movie and TV vocabularies differ slightly)
movie_genres = {g["id"]: g["name"] for g in tmdb_get("/genre/movie/list")["genres"]}
tv_genres = {g["id"]: g["name"] for g in tmdb_get("/genre/tv/list")["genres"]}
print(f"{len(movie_genres)} movie genres, {len(tv_genres)} TV genres")


In [ ]:
# Discover loop, sliced by year.
# /discover caps at 500 pages per query; per-year slices stay far below it.

def discover_year(media_type, year):
    rows, page, total_pages = [], 1, 1
    year_param = (
        {"primary_release_year": year} if media_type == "movie"
        else {"first_air_date_year": year}
    )
    genre_map = movie_genres if media_type == "movie" else tv_genres
    while page <= total_pages:
        payload = tmdb_get(
            f"/discover/{media_type}",
            sort_by="vote_count.desc",
            **{"vote_count.gte": VOTE_COUNT_MIN},
            include_adult="false",
            page=page,
            **year_param,
        )
        total_pages = min(payload["total_pages"], 500)
        for item in payload["results"]:
            title = item.get("title") or item.get("name") or ""
            date = item.get("release_date") or item.get("first_air_date") or ""
            if not title or len(date) < 4:
                continue
            rows.append({
                "tmdbKey": f"{media_type}-{item['id']}",
                "tmdbId": item["id"],
                "mediaType": media_type,
                "title": title[:500],
                "releaseYear": int(date[:4]),
                "genres": ",".join(genre_map.get(g, "") for g in item.get("genre_ids", []) if g in genre_map),
                "voteAverage": float(item.get("vote_average") or 0.0),
                "voteCount": int(item.get("vote_count") or 0),
                "popularity": float(item.get("popularity") or 0.0),
                "posterPath": item.get("poster_path"),
                "originalLanguage": (item.get("original_language") or "")[:10],
            })
        page += 1
        time.sleep(0.03)
    return rows

all_rows = []
for media_type in ("movie", "tv"):
    for year in range(YEAR_FROM, YEAR_TO + 1):
        batch = discover_year(media_type, year)
        all_rows.extend(batch)
    print(f"{media_type}: cumulative {len(all_rows)} rows")


In [ ]:
# Runtime enrichment is skipped at discover level (not in the payload).
# Movie runtimes arrive with the credits fetch in Notebook 02, which calls
# /movie/{id} append_to_response=credits in a single request per title.

import pandas as pd

df = pd.DataFrame(all_rows).drop_duplicates(subset=["tmdbKey"])
df["decade"] = (df["releaseYear"] // 10) * 10
df["isSeries"] = df["mediaType"].eq("tv")

# Scope validation: title counts by vote threshold, so the cut is evidenced
for threshold in (200, 500, 1000, 5000):
    print(f"vote_count >= {threshold}: {(df['voteCount'] >= threshold).sum():,} titles")
print(df.groupby('mediaType').size())


In [ ]:
# Write raw_titles Delta (full overwrite - this is a one-shot bulk load)
sdf = spark.createDataFrame(df)
sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("raw_titles")
print(f"raw_titles written: {sdf.count():,} rows")
